# Random-Direction Control: does the steering vector's direction matter at all?

## The gap this closes

This project's central mechanistic claim is **magnitude dominates direction**: how hard you push
matters, which way you push does not. The evidence for it so far is convergence -- four
independently-constructed contrastive vectors (naive 12-example, 68-example scaled, 40-pair
matched, 150-pair strength-matched, plus the utility-matched fair-test vector) all produce
similar collapse once compared at matched push strength.

That is suggestive but it is not the test. **Every one of those vectors points in roughly the
same direction** -- they are all built to separate repetitive from non-repetitive protein. A
reviewer's cheapest reply is: "of course they agree, they are all repetition vectors; that shows
your constructions are equivalent, not that direction is irrelevant."

The claim has never been tested against a direction that is not a steering vector at all. This
notebook does that.

## Design

Everything is held fixed at this project's most-cited operating point: **ProtGPT2, layer 12,
`REFERENCE_NORM = 583.998`, N=50/condition, `temperature=1.2`, `max_len=50`.** The real vector is
rebuilt from the exact same hand-picked contrastive sets as `03-ai4dd-uccs-baseline-test.ipynb`,
so `REAL_1x`/`REAL_2x` should reproduce that notebook's locked numbers (§1a: CONTROL 62.0% ->
1x 60.0% -> 2x 100.0%) and act as an internal calibration check. If they do not reproduce, that
is itself worth knowing before reading anything else in the table.

Against that, four different ways of *not* being the real direction, all at identical norm:

| Condition | What it is | What it controls for |
|---|---|---|
| `RANDOM` | fresh Gaussian vector, rescaled to the same norm | is any push of this size enough? |
| `SHUFFLED` | the real `v_L` with its 1280 coordinates permuted | identical norm **and** identical coordinate distribution -- only the direction is destroyed. The tightest control of the four. |
| `ORTHOGONAL` | random vector with its `v_L` component projected out | guaranteed 90° to the real direction -- "maximally not the steering direction" |
| `NEGATED` | exactly `-v_L` | steering *toward* repetition instead of away. If this also collapses structure, direction is not merely unimportant, it is irrelevant. |

`RANDOM` is tested at both 1x and 2x; the others at 2x only, since 2x is where the real effect
lives (nothing measurable happens at 1x for any vector on ProtGPT2, per §1a and §1g) and folding
budget is better spent on more *kinds* of wrong direction than on more doses of one.

Eight conditions × 50 = 400 folds, comparable to `24`'s total and cheaper than `08`'s.

## How to read the result

- **Random/shuffled/orthogonal collapse ~as hard as REAL at 2x** -> the claim is demonstrated,
  not merely suggested, and it sharpens considerably: the method's *content* is doing no work.
  What PEP calls a carefully-constructed contrastive direction behaves like noise of the same
  size. That is a much stronger cautionary result than "steering hurts structure".
- **Random collapses noticeably less than REAL** -> direction does carry real information, and
  the "magnitude dominates" claim must be narrowed to something like "magnitude dominates *among
  plausible steering directions*". Also a genuine, reportable result -- and better found now than
  in review.
- **`NEGATED` matters on its own.** If pushing toward repetition damages structure as much as
  pushing away from it, no directional story survives at all.

There is no outcome here that wastes the run.

## Two things this notebook also fixes in passing

1. It records the **mean residual-stream norm ‖h‖ at layer 12**, so every condition can be
   reported in `alpha_rel` units (see `34-ai4dd-residual-norm-audit.ipynb` and
   `notes/locked-results.md` §1k-FLAG) rather than only as a bare multiplier.
2. It **saves every generated sequence to CSV** with its condition, pLDDT, pTM, entropy and
   length. No notebook in this project except `00a` currently does this, which is why sequence-
   level questions (what does collapse actually look like? are collapsed sequences shorter?)
   have never been answerable without a fresh GPU run.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~60-90 minutes.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"

REFERENCE_NORM = 583.998   # same shared reference as 24/26/27/28/32
TARGET_LAYER = 12          # this project's most-cited injection point
N_PER_CONDITION = 50

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [2]:
# --- Rebuild the naive v_L from the EXACT contrastive sets used in
#     03-ai4dd-uccs-baseline-test.ipynb. Reusing them verbatim (rather than rebuilding a
#     utility-matched vector, which would need a 200-sequence fold first) is deliberate: it
#     makes REAL_1x/REAL_2x in this notebook directly comparable to the locked §1a numbers,
#     giving a free internal calibration check. This vector's natural norm is already ~583.998
#     -- it is where REFERENCE_NORM came from in the first place. ---

positive_seqs = [
    "NLYIQWLKDGGPSSGRPPPS", "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF", "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]
degenerate_seqs = [
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA", "LGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGL",
    "GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG", "SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS",
    "PGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGP", "QWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQ",
]

print(f"Loading ProtGPT2 on {device}...")
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

pos_acts = get_mean_activation(plm_model, tokenizer, positive_seqs, TARGET_LAYER)
neg_acts = get_mean_activation(plm_model, tokenizer, degenerate_seqs, TARGET_LAYER)
v_L_raw = (pos_acts.mean(dim=0) - neg_acts.mean(dim=0))
raw_norm = v_L_raw.norm().item()
v_L = (v_L_raw * (REFERENCE_NORM / raw_norm)).to(device)

print(f"Naive v_L raw norm: {raw_norm:.4f}  -> normalized to {v_L.norm().item():.4f}")
print(f"(03-ai4dd-uccs-baseline-test.ipynb recorded 583.9979 for this same construction --")
print(f" a close match here confirms the vector is being rebuilt faithfully.)")


Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naive v_L raw norm: 583.9979  -> normalized to 583.9980
(03-ai4dd-uccs-baseline-test.ipynb recorded 583.9979 for this same construction --
 a close match here confirms the vector is being rebuilt faithfully.)


In [3]:
# --- Measure the residual-stream norm at the injection site, so every condition below can
#     also be reported in alpha_rel = ‖v‖/‖h‖ units. Uses a forward hook on block TARGET_LAYER,
#     i.e. exactly the tensor the steering hook modifies. Cheap -- forward passes only. ---

def measure_resid_norm(model, tokenizer, seqs, layer):
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = model.transformer.h[layer].register_forward_hook(hook)
    per_pos = []
    try:
        for seq in seqs:
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
            captured.clear()
            with torch.no_grad():
                model(**inputs)
            if "h" in captured:
                per_pos.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(per_pos)) if per_pos else float("nan")

resid_norm = measure_resid_norm(plm_model, tokenizer, positive_seqs + degenerate_seqs, TARGET_LAYER)
alpha_rel_1x = REFERENCE_NORM / resid_norm
print(f"Mean residual-stream norm ‖h‖ at layer {TARGET_LAYER}: {resid_norm:.2f}")
print(f"  => alpha_rel at '1x' = {alpha_rel_1x:.4f}   (at '2x' = {2 * alpha_rel_1x:.4f})")
print(f"  i.e. '1x' displaces each token's hidden state by ~{alpha_rel_1x:.1%} of its own magnitude.")


Mean residual-stream norm ‖h‖ at layer 12: 3397.13
  => alpha_rel at '1x' = 0.1719   (at '2x' = 0.3438)
  i.e. '1x' displaces each token's hidden state by ~17.2% of its own magnitude.


In [4]:
# --- Build the four not-the-real-direction vectors. Every one is rescaled to exactly
#     REFERENCE_NORM, so the ONLY thing that differs from REAL is which way it points. ---

gen = torch.Generator(device="cpu").manual_seed(31337)
d_model = v_L.shape[0]

def to_ref_norm(v):
    return (v * (REFERENCE_NORM / v.norm())).to(device)

# 1. RANDOM -- fresh isotropic Gaussian direction.
v_random = to_ref_norm(torch.randn(d_model, generator=gen))

# 2. SHUFFLED -- v_L's own coordinates, permuted. Identical norm AND identical multiset of
#    coordinate values; only the assignment of values to dimensions changes. This is the
#    tightest control available: it cannot be dismissed as "your random vector had different
#    statistics from a real steering vector".
perm = torch.randperm(d_model, generator=gen)
v_shuffled = to_ref_norm(v_L.detach().cpu()[perm])

# 3. ORTHOGONAL -- random direction with the v_L component projected out, so it is provably
#    at 90 degrees to the real steering direction.
g = torch.randn(d_model, generator=gen)
v_cpu = v_L.detach().cpu()
g_orth = g - (torch.dot(g, v_cpu) / torch.dot(v_cpu, v_cpu)) * v_cpu
v_orthogonal = to_ref_norm(g_orth)

# 4. NEGATED -- exactly the opposite of the real direction: steer TOWARD repetition.
v_negated = to_ref_norm(-v_cpu)

def cos_sim(a, b):
    a, b = a.detach().cpu().float(), b.detach().cpu().float()
    return float(torch.dot(a, b) / (a.norm() * b.norm()))

print("Direction check -- cosine similarity to the real v_L (1.0 = identical direction):")
print(f"  REAL       : {cos_sim(v_L, v_L):+.4f}")
print(f"  RANDOM     : {cos_sim(v_random, v_L):+.4f}   (expect ~0 in 1280 dimensions)")
print(f"  SHUFFLED   : {cos_sim(v_shuffled, v_L):+.4f}")
print(f"  ORTHOGONAL : {cos_sim(v_orthogonal, v_L):+.4f}   (expect ~0 by construction)")
print(f"  NEGATED    : {cos_sim(v_negated, v_L):+.4f}   (expect exactly -1)")
print()
print("Norm check -- all must equal REFERENCE_NORM, or the comparison is not controlled:")
for nm, v in [("REAL", v_L), ("RANDOM", v_random), ("SHUFFLED", v_shuffled),
              ("ORTHOGONAL", v_orthogonal), ("NEGATED", v_negated)]:
    print(f"  {nm:11s}: {v.norm().item():.4f}")


Direction check -- cosine similarity to the real v_L (1.0 = identical direction):
  REAL       : +1.0000
  RANDOM     : +0.0394   (expect ~0 in 1280 dimensions)
  SHUFFLED   : -0.0042
  ORTHOGONAL : -0.0000   (expect ~0 by construction)
  NEGATED    : -1.0000   (expect exactly -1)

Norm check -- all must equal REFERENCE_NORM, or the comparison is not controlled:
  REAL       : 583.9980
  RANDOM     : 583.9980
  SHUFFLED   : 583.9979
  ORTHOGONAL : 583.9980
  NEGATED    : 583.9980


In [5]:
# --- Same live-UniProt prefix pool and same steering hook as every other notebook in the
#     project. Nothing about the injection mechanism changes here -- only the vector. ---

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print()
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING.")
    print("!!")
    print("!! A 'name resolution' error here means Kaggle's Internet toggle is OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!!")
    print("!! With Internet off the HuggingFace downloads will also fail unless cached.")
    print("!! Turning Internet on is the real fix. This fallback only prevents a crash and")
    print("!! covers a transient UniProt outage.")
    print("!!")
    print("!! Falling back to the real protein fragments already hardcoded above")
    print("!! (03-ai4dd-uccs-baseline-test.ipynb's positive_seqs -- real, not synthetic).")
    print("!! Prefixes will be drawn from 6 source proteins instead of 12, which reduces")
    print("!! prompt diversity. Note that on any result taken from such a run.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(positive_seqs)]

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError(
            "No reference sequences available -- neither the UniProt fetch nor the fallback "
            "produced anything. Check Kaggle's Internet setting before re-running."
        )
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, prompts,
                                  max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = model.transformer.h[target_layer].register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        handle.remove()
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_part = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_part,
                        "entropy": calculate_entropy(gen_part)})
    clear_gpu()
    return records

steer_prefixes = build_prefix_pool(reference_seqs, n_prefixes=450, seed=23)
print(f"\nBuilt {len(steer_prefixes)} calibration prefixes.")

# Each condition gets its own disjoint slice of prefixes and its own seed, matching the
# convention used in 24/26/27/28/32.
CONDITION_SPECS = [
    ("CONTROL",      None,                111),
    ("REAL_1x",      v_L * 1.0,           222),
    ("REAL_2x",      v_L * 2.0,           333),
    ("RANDOM_1x",    v_random * 1.0,      444),
    ("RANDOM_2x",    v_random * 2.0,      555),
    ("SHUFFLED_2x",  v_shuffled * 2.0,    666),
    ("ORTHOGONAL_2x", v_orthogonal * 2.0, 777),
    ("NEGATED_2x",   v_negated * 2.0,     888),
]

conditions = {}
for i, (name, vec, seed) in enumerate(CONDITION_SPECS):
    lo = i * N_PER_CONDITION
    hi = lo + N_PER_CONDITION
    print(f"=== {name} (prefixes {lo}:{hi}, seed {seed}) ===")
    conditions[name] = generate_with_vector_steering(
        plm_model, tokenizer, TARGET_LAYER, vec, steer_prefixes[lo:hi], seed=seed)

print("\n=== Freeing ProtGPT2 from GPU ===")
del plm_model
clear_gpu()


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 450 calibration prefixes.
=== CONTROL (prefixes 0:50, seed 111) ===
=== REAL_1x (prefixes 50:100, seed 222) ===
=== REAL_2x (prefixes 100:150, seed 333) ===
=== RANDOM_1x (prefixes 150:200, seed 444) ===
=== RANDOM_2x (prefixes 200:250, seed 555) ===
=== SHUFFLED_2x (prefixes 250:300, seed 666) ===
=== ORTHOGONAL_2x (prefixes 300:350, seed 777) ===
=== NEGATED_2x (prefixes 350:400, seed 888) ===

=== Freeing ProtGPT2 from GPU ===


In [6]:
# --- Fold everything. Same VALID_AA filter and same pLDDT<60 collapse threshold as every
#     other notebook in the project. ---

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0
        inputs = self.tokenizer([cleaned], return_tensors="pt", add_special_tokens=False).to(self.device)
        plddt, ptm = 0.0, 0.0
        try:
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
        except RuntimeError:
            clear_gpu()
        return plddt, ptm

evaluator = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    for r in conditions[name]:
        plddt, ptm = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["collapse"] = int(0.0 < plddt < 60.0)
del evaluator
clear_gpu()

print(f"\n{'Condition':16s} {'N':>4s} {'Entropy':>9s} {'pLDDT':>8s} {'pTM':>7s} {'Collapse%':>10s}")
print("-" * 60)
for name, recs in conditions.items():
    ents = [r["entropy"] for r in recs]
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0.0]
    ptms = [r["ptm"] for r in recs if r["plddt"] > 0.0]
    collapse_rate = np.mean([r["collapse"] for r in recs])
    print(f"{name:16s} {len(recs):4d} {np.mean(ents):9.3f} {np.mean(plddts):8.2f} "
          f"{np.mean(ptms):7.3f} {collapse_rate * 100:9.1f}%")


Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding REAL_1x...
Folding REAL_2x...
Folding RANDOM_1x...
Folding RANDOM_2x...
Folding SHUFFLED_2x...
Folding ORTHOGONAL_2x...
Folding NEGATED_2x...

Condition           N   Entropy    pLDDT     pTM  Collapse%
------------------------------------------------------------
CONTROL            50     2.847    59.17   0.296      52.0%
REAL_1x            50     2.804    54.59   0.165      64.0%
REAL_2x            50     4.061    30.92   0.158     100.0%
RANDOM_1x          50     3.209    41.09   0.165      92.0%
RANDOM_2x          50     4.089    30.98   0.157     100.0%
SHUFFLED_2x        50     4.000    31.45   0.158     100.0%
ORTHOGONAL_2x      50     4.010    37.86   0.241       2.0%
NEGATED_2x         50     1.143    49.35   0.111      70.0%


In [7]:
# --- Analysis: CIs, and the two comparisons this notebook exists to make. ---
from scipy.stats import fisher_exact

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

summary = {}
for name, recs in conditions.items():
    k = int(np.sum([r["collapse"] for r in recs]))
    n = len(recs)
    lo, hi = wilson_ci(k, n)
    summary[name] = {"k": k, "n": n, "rate": k / n, "ci": (lo, hi)}

print("=" * 92)
print("COLLAPSE RATES WITH 95% WILSON CIs")
print("=" * 92)
print(f"{'Condition':16s} {'collapsed':>12s} {'rate':>8s} {'95% CI':>22s} {'alpha_rel':>10s}")
print("-" * 92)
for name in conditions:
    s = summary[name]
    mult = 0.0 if name == "CONTROL" else (2.0 if name.endswith("2x") else 1.0)
    ci_lo, ci_hi = s["ci"]
    ci_str = "[{:.1%}, {:.1%}]".format(ci_lo, ci_hi)
    print(f"{name:16s} {s['k']:6d}/{s['n']:<5d} {s['rate']:7.1%} {ci_str:>22s} "
          f"{mult * alpha_rel_1x:10.4f}")

print()
print("=" * 92)
print("CALIBRATION CHECK -- do REAL_1x / REAL_2x reproduce locked §1a?")
print("=" * 92)
print("locked §1a (03-ai4dd-uccs-baseline-test): CONTROL 62.0% -> 1x 60.0% -> 2x 100.0%")
print(f"this run:                                 CONTROL {summary['CONTROL']['rate']:.1%} -> "
      f"1x {summary['REAL_1x']['rate']:.1%} -> 2x {summary['REAL_2x']['rate']:.1%}")
print()
print("If these are far apart, stop and investigate before reading anything below -- the")
print("comparison logic assumes this run reproduces the established reference point.")

print()
print("=" * 92)
print("THE ACTUAL TEST -- every 2x condition against REAL_2x, and against CONTROL")
print("=" * 92)
print(f"{'Condition':16s} {'rate':>8s} {'vs REAL_2x (Fisher p)':>24s} {'vs CONTROL (Fisher p)':>24s}")
print("-" * 92)

real2 = summary["REAL_2x"]
ctrl = summary["CONTROL"]
for name in ["REAL_2x", "RANDOM_2x", "SHUFFLED_2x", "ORTHOGONAL_2x", "NEGATED_2x"]:
    s = summary[name]
    if name == "REAL_2x":
        p_real = float("nan")
    else:
        _, p_real = fisher_exact([[s["k"], s["n"] - s["k"]],
                                  [real2["k"], real2["n"] - real2["k"]]])
    _, p_ctrl = fisher_exact([[s["k"], s["n"] - s["k"]],
                              [ctrl["k"], ctrl["n"] - ctrl["k"]]])
    p_real_str = "--" if name == "REAL_2x" else f"{p_real:.4f}"
    print(f"{name:16s} {s['rate']:7.1%} {p_real_str:>24s} {p_ctrl:>23.4f}")

print()
print("=" * 92)
print("VERDICT")
print("=" * 92)

wrong_dirs = ["RANDOM_2x", "SHUFFLED_2x", "ORTHOGONAL_2x", "NEGATED_2x"]
n_matching_real = 0
n_above_control = 0
for name in wrong_dirs:
    s = summary[name]
    _, p_real = fisher_exact([[s["k"], s["n"] - s["k"]], [real2["k"], real2["n"] - real2["k"]]])
    _, p_ctrl = fisher_exact([[s["k"], s["n"] - s["k"]], [ctrl["k"], ctrl["n"] - ctrl["k"]]])
    if p_real > 0.05:
        n_matching_real += 1
    if p_ctrl < 0.05 and s["rate"] > ctrl["rate"]:
        n_above_control += 1

print(f"Wrong-direction conditions statistically indistinguishable from REAL_2x: "
      f"{n_matching_real}/{len(wrong_dirs)}")
print(f"Wrong-direction conditions significantly worse than CONTROL:             "
      f"{n_above_control}/{len(wrong_dirs)}")
print()
if n_matching_real >= 3 and n_above_control >= 3:
    print("  ==> MAGNITUDE DOMINATES DIRECTION -- now demonstrated, not just suggested.")
    print("      Pushes with no relationship to the repetition direction (including one that")
    print("      points the exact opposite way) damage structure as much as the carefully-")
    print("      constructed contrastive vector does. The contrastive construction contributes")
    print("      nothing detectable beyond its magnitude.")
    print()
    print("      This substantially strengthens the paper's central claim and should be")
    print("      promoted to a headline result, not filed as a control.")
elif n_above_control >= 3 and n_matching_real <= 1:
    print("  ==> DIRECTION CARRIES REAL INFORMATION. Wrong directions do damage structure, but")
    print("      measurably less than the real vector. The 'magnitude dominates' claim must be")
    print("      narrowed -- something like 'magnitude dominates among plausible steering")
    print("      directions' -- and the paper should report the gap explicitly rather than")
    print("      keeping the stronger phrasing.")
else:
    print("  ==> MIXED. Read the per-condition table above directly; the pattern does not reduce")
    print("      to a one-line verdict. Note especially whether NEGATED_2x behaves like REAL_2x")
    print("      (no directional story survives) or unlike it (direction matters, sign matters).")


COLLAPSE RATES WITH 95% WILSON CIs
Condition           collapsed     rate                 95% CI  alpha_rel
--------------------------------------------------------------------------------------------
CONTROL              26/50      52.0%         [38.5%, 65.2%]     0.0000
REAL_1x              32/50      64.0%         [50.1%, 75.9%]     0.1719
REAL_2x              50/50     100.0%        [92.9%, 100.0%]     0.3438
RANDOM_1x            46/50      92.0%         [81.2%, 96.8%]     0.1719
RANDOM_2x            50/50     100.0%        [92.9%, 100.0%]     0.3438
SHUFFLED_2x          50/50     100.0%        [92.9%, 100.0%]     0.3438
ORTHOGONAL_2x         1/50       2.0%          [0.4%, 10.5%]     0.3438
NEGATED_2x           35/50      70.0%         [56.2%, 80.9%]     0.3438

CALIBRATION CHECK -- do REAL_1x / REAL_2x reproduce locked §1a?
locked §1a (03-ai4dd-uccs-baseline-test): CONTROL 62.0% -> 1x 60.0% -> 2x 100.0%
this run:                                 CONTROL 52.0% -> 1x 64.0% -> 2x 100

In [8]:
# --- Persist everything. This is the fix for the standing data-loss problem: no notebook in
#     this project except 00a saves its generated sequences, so every sequence-level question
#     (failure-mode taxonomy, length confound, ESM-2 second opinion, eyeballing examples) has
#     needed a fresh GPU run to answer. From here on it does not. ---

rows = []
for name, recs in conditions.items():
    for i, r in enumerate(recs):
        rows.append({
            "condition": name,
            "idx": i,
            "prompt": r["prompt"],
            "sequence": r["sequence"],
            "gen_only": r["gen_only"],
            "gen_length": len(r["gen_only"]),
            "usable_length": sum(1 for a in r["gen_only"] if a in VALID_AA),
            "entropy": r["entropy"],
            "plddt": r["plddt"],
            "ptm": r["ptm"],
            "collapse": r["collapse"],
        })

seq_df = pd.DataFrame(rows)
seq_df.to_csv("random_direction_control_sequences.csv", index=False)

summary_rows = []
for name, s in summary.items():
    mult = 0.0 if name == "CONTROL" else (2.0 if name.endswith("2x") else 1.0)
    summary_rows.append({
        "condition": name, "collapsed": s["k"], "n": s["n"], "collapse_rate": s["rate"],
        "ci_lo": s["ci"][0], "ci_hi": s["ci"][1],
        "multiplier": mult, "alpha_rel": mult * alpha_rel_1x,
        "resid_norm_h": resid_norm, "reference_norm": REFERENCE_NORM, "layer": TARGET_LAYER,
    })
pd.DataFrame(summary_rows).to_csv("random_direction_control_summary.csv", index=False)

print("Saved:")
print(f"  random_direction_control_sequences.csv  ({len(seq_df)} sequences, all conditions)")
print(f"  random_direction_control_summary.csv    ({len(summary_rows)} conditions)")
print()
print("Length sanity check by condition (a confound never yet examined in this project --")
print("if steered sequences are systematically shorter, part of 'collapse' may be a length effect):")
print()
print(f"{'Condition':16s} {'mean gen len':>13s} {'mean usable':>13s} {'mean pLDDT':>12s}")
print("-" * 58)
for name in conditions:
    sub = seq_df[seq_df["condition"] == name]
    print(f"{name:16s} {sub['gen_length'].mean():13.1f} {sub['usable_length'].mean():13.1f} "
          f"{sub['plddt'].mean():12.2f}")
print()
print("Download both CSVs from Kaggle's output pane and commit them next to this notebook.")


Saved:
  random_direction_control_sequences.csv  (400 sequences, all conditions)
  random_direction_control_summary.csv    (8 conditions)

Length sanity check by condition (a confound never yet examined in this project --
if steered sequences are systematically shorter, part of 'collapse' may be a length effect):

Condition         mean gen len   mean usable   mean pLDDT
----------------------------------------------------------
CONTROL                   56.5          54.7        59.17
REAL_1x                   42.7          30.5        54.59
REAL_2x                  192.6         176.6        30.92
RANDOM_1x                 95.8          61.0        41.09
RANDOM_2x                187.0         183.9        30.98
SHUFFLED_2x              198.0         196.9        31.45
ORTHOGONAL_2x            270.1         268.6         0.76
NEGATED_2x                61.2          58.6        48.36

Download both CSVs from Kaggle's output pane and commit them next to this notebook.
